# Trabajo Práctico: Predicción de precios de casas

**Aprendizaje Automático 1 — Tecnicatura Universitaria en Inteligencia Artificial (FCEIA - UNR)**

**Integrantes:** Franco Esparza, Franco Renna, Leandro Picó

**Objetivo:** predecir `MEDV` (valor mediano de las viviendas de cada barrio de Boston, en miles de dólares) a partir de 13 características del barrio.

### Contenidos
1. Configuración
2. Carga de datos
3. Datos faltantes y consistencia
4. Limpieza previa al split
5. División train / validación / test

---
## 1. Configuración

In [ ]:
# Importación de librerías y configuración global de gráficos y de pandas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import warnings
warnings.filterwarnings('ignore')

# Configuracion global de visualizacion
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.dpi': 100
})
sns.set_theme(style='whitegrid', palette='viridis')

# Configuracion de pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print('Entorno configurado correctamente.')

Entorno configurado correctamente.


---
## 2. Carga de datos

El target `MEDV` es un número continuo, así que el problema es de **regresión**. Las 13 variables explicativas se describen en la sección 6.

In [ ]:
# Carga del dataset y definicion del target y de las variables explicativas
df = pd.read_csv('house-prices-tp.csv')

TARGET = 'MEDV'
cols_features = [c for c in df.columns if c != TARGET]

print(f'Dimensiones del dataset: {df.shape[0]} filas x {df.shape[1]} columnas')
df.head()

Dimensiones del dataset: 556 filas x 14 columnas


,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.0715,0.0000,4.4900,0.0000,0.4490,6.1210,56.8000,3.7476,3.0000,247.0000,18.5000,395.1500,8.4400,22.2000
1,0.0827,0.0000,13.9200,0.0000,0.4370,6.1270,18.4000,5.5027,4.0000,289.0000,16.0000,396.9000,8.5800,23.9000
2,0.1282,12.5000,6.0700,0.0000,0.4090,5.8850,33.0000,6.4980,4.0000,345.0000,18.9000,396.9000,8.7900,20.9000
3,0.0887,21.0000,5.6400,0.0000,0.4390,5.9630,45.7000,6.8147,4.0000,243.0000,16.8000,395.5600,13.4500,19.7000
4,0.1143,0.0000,8.5600,0.0000,0.5200,6.7810,71.3000,2.8561,5.0000,384.0000,20.9000,395.5800,7.6700,26.5000


---
## 3. Datos faltantes y consistencia

Se analiza sobre todo el dataset porque acá solo se cuenta: no se calcula nada que después se use en el modelo.

In [ ]:
# Reporte de valores nulos por columna
nulos = df.isnull().sum()
pct_nulos = (nulos / len(df)) * 100

reporte_nulos = pd.DataFrame({
    'Nulos': nulos,
    'Porcentaje (%)': pct_nulos,
    'Tipo': df.dtypes
})

print('Reporte de Valores Nulos:')
print('=' * 50)
print(reporte_nulos)
print(f'\nTotal de celdas con nulos: {nulos.sum()} '
      f'({nulos.sum() / df.size * 100:.4f}% del dataset)')

Reporte de Valores Nulos:
         Nulos  Porcentaje (%)     Tipo
CRIM        23          4.1367  float64
ZN          22          3.9568  float64
INDUS       15          2.6978  float64
CHAS        23          4.1367  float64
NOX         24          4.3165  float64
RM          21          3.7770  float64
AGE         24          4.3165  float64
DIS         15          2.6978  float64
RAD         28          5.0360  float64
TAX         18          3.2374  float64
PTRATIO     28          5.0360  float64
B           22          3.9568  float64
LSTAT       22          3.9568  float64
MEDV        21          3.7770  float64

Total de celdas con nulos: 306 (3.9311% del dataset)


Todas las columnas tienen entre 2,7% y 5% de faltantes, incluido el target. Veamos cómo se reparten entre las filas.

In [ ]:
# Faltantes por fila: ¿estan repartidos en muchas filas o concentrados en pocas?
nulos_por_fila = df.isna().sum(axis=1)
p_faltante = df.isna().mean().mean()
filas_esperadas = len(df) * (1 - (1 - p_faltante) ** df.shape[1])  # si faltaran al azar e independientes

print(f'Filas completas: {(nulos_por_fila == 0).sum()}')
print(f'Filas con al menos un faltante: {(nulos_por_fila > 0).sum()} ({(nulos_por_fila > 0).mean() * 100:.2f}%)')
print(f'Filas incompletas esperadas si los faltantes fueran al azar: {filas_esperadas:.0f}')

Filas completas: 506
Filas con al menos un faltante: 50 (8.99%)
Filas incompletas esperadas si los faltantes fueran al azar: 239


In [ ]:
# Duplicados, valores fuera del dominio posible y valores no enteros de RAD (es un indice, deberia ser entero)
rad_no_entero = df['RAD'].notna() & (df['RAD'] % 1 != 0)

print(f'Registros duplicados: {df.duplicated().sum()}')
print(f'Valores negativos: {(df < 0).sum().sum()}')
print(f'Porcentajes (ZN, INDUS, AGE, LSTAT) fuera de [0, 100]: '
      f'{((df[["ZN", "INDUS", "AGE", "LSTAT"]] < 0) | (df[["ZN", "INDUS", "AGE", "LSTAT"]] > 100)).sum().sum()}')
print(f'Valores distintos de CHAS: {sorted(df["CHAS"].dropna().unique().tolist())}')
print(f'Valores no enteros de RAD: {rad_no_entero.sum()} '
      f'(en filas completas: {(rad_no_entero & (nulos_por_fila == 0)).sum()} | '
      f'en filas con faltantes: {(rad_no_entero & (nulos_por_fila > 0)).sum()})')

Registros duplicados: 0
Valores negativos: 0
Porcentajes (ZN, INDUS, AGE, LSTAT) fuera de [0, 100]: 0
Valores distintos de CHAS: [0.0, 1.0]
Valores no enteros de RAD: 22 (en filas completas: 0 | en filas con faltantes: 22)


**Conclusiones y decisiones:**
- Los faltantes están **concentrados**: si fueran al azar habría más de 200 filas incompletas, pero hay solo **50**, y el resto (506) está completo. En la Figura 3 (sección 7) se ve además que esas 50 filas no siguen el patrón de las demás.
- **Se conservan**, porque eliminarlas sería perder el 9% de los datos. Adoptamos como criterio no descartar más del 5%. Sus faltantes se imputan en la sección 9.
- No hay duplicados ni valores fuera de rango. La única inconsistencia son los **valores no enteros de `RAD`**: es un índice, en las filas completas siempre es entero, y todos los no enteros están en las filas con faltantes.

---
## 4. Limpieza previa al split

Estas dos correcciones se deciden **fila por fila**, sin calcular nada con el resto de los datos, así que pueden hacerse antes de dividir sin fuga de información:
- **Filas sin `MEDV`:** se eliminan. Sin el valor real no sirven para entrenar ni para evaluar, e imputar el target sería inventar la respuesta. Son el 3,8%, dentro del 5%.
- **`RAD` no enteros:** se redondean al entero más cercano.

In [ ]:
# Eliminacion de filas sin target y redondeo de los valores no enteros de RAD
filas_antes = df.shape[0]
df = df[df[TARGET].notna()].reset_index(drop=True)
print(f'Filas eliminadas por no tener MEDV: {filas_antes - df.shape[0]} '
      f'({(filas_antes - df.shape[0]) / filas_antes * 100:.2f}%) -> quedan {df.shape[0]}')

rad_no_entero = df['RAD'].notna() & (df['RAD'] % 1 != 0)
df.loc[rad_no_entero, 'RAD'] = df.loc[rad_no_entero, 'RAD'].round()
print(f'Valores de RAD redondeados: {rad_no_entero.sum()} | no enteros restantes: {(df["RAD"].dropna() % 1 != 0).sum()}')

Filas eliminadas por no tener MEDV: 21 (3.78%) -> quedan 535
Valores de RAD redondeados: 17 | no enteros restantes: 0


---
## 5. División train / validación / test

Dividimos **antes** de analizar correlaciones, imputar y escalar, para que ninguna de esas decisiones use información de validación o test.
- **Train (70%):** entrenar y hacer el análisis.
- **Validación (10%):** elegir hiperparámetros sin tocar el test.
- **Test (20%):** evaluar al final.

In [ ]:
# Division en dos pasos: 20% test y, del 80% restante, 12,5% validacion (= 10% del total)
X = df[cols_features]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.125, random_state=42)

X_train.shape, X_val.shape, X_test.shape

((374, 13), (54, 13), (107, 13))

In [ ]:
# Union de X_train e y_train para el analisis exploratorio (solo train)
data_train = pd.concat([X_train, y_train], axis=1)
filas_con_faltantes = X_train.isna().any(axis=1)
print(f'Filas de train con algun faltante: {filas_con_faltantes.sum()} de {len(X_train)}')

Filas de train con algun faltante: 20 de 374
